# CSEE 4121 Workshop: RAG

In this workshop, you will build a Retrieval-Augmented Generation pipeline from scratch using NumPy, a sentence embedding model, and an LLM. Use a T4 runtime to complete.

### Learning objectives

1. What each stage of a RAG pipeline does
2. How embeddings turn text into vectors that encode meaning
3. How brute-force similarity search works
4. Where the time goes in a RAG query
5. Why systems need vector indices

By the end, you'll have a working RAG system over a corpus of US history Wikipedia articles, and you'll have measured empirically why a brute-force linear scan over 1.4 billion vectors needs a proper vector index.

**Before you start**, set the runtime to GPU. `Runtime → Change runtime type → T4 GPU`.

## Section 0: Setup

Install dependencies and download models. This cell takes ~2 minutes on first run. Afterwards, the models are cached.

In [ ]:
# Install dependencies (quiet)
!pip install -q sentence-transformers transformers accelerate wikipedia-api

import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
assert DEVICE == "cuda", "Enable GPU: Runtime → Change runtime type → T4 GPU"


In [ ]:
# Load the embedding model (small, fast, ~90MB)
# all-MiniLM-L6-v2 maps sentences to 384-dim vectors
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")


In [ ]:
# Load the generator model: TinyLlama-1.1B-Chat
# Small enough for a T4, chat-tuned so it follows instructions
GEN_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Loaded {GEN_MODEL}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")


In [ ]:
# Helper: generate text from a prompt using TinyLlama's chat template
def generate(prompt: str, max_new_tokens: int = 150) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs, # Unpack the dictionary to pass input_ids and attention_mask
            max_new_tokens=max_new_tokens,
            do_sample=False,   # greedy for reproducibility
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Strip the prompt tokens; return only the new text
    new_tokens = output[0, inputs["input_ids"].shape[1]:] # Access shape from input_ids tensor
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Smoke test
print(generate("In one sentence, what is 2+2?"))

### Download the corpus

We'll use Wikipedia articles on US history topics. The code below pulls ~12 articles and concatenates them into a single text blob. This is our "private knowledge base" that the LLM wasn't necessarily trained on in full detail.

In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent="CSEE4121-Workshop/1.0 (educational)",
    language="en",
)

ARTICLES = [
    "History of the United States",
    "American Revolution",
    "Constitutional Convention (United States)",
    "Louisiana Purchase",
    "American Civil War",
    "Reconstruction era",
    "Gilded Age",
    "Progressive Era",
    "Great Depression",
    "New Deal",
    "Civil rights movement",
    "Watergate scandal",
]

corpus_texts = {}
for title in ARTICLES:
    page = wiki.page(title)
    if page.exists():
        corpus_texts[title] = page.text
        print(f"  Y {title}: {len(page.text):,} chars")
    else:
        print(f"  N {title}: not found")

total_chars = sum(len(t) for t in corpus_texts.values())
print(f"\nTotal corpus: {total_chars:,} characters across {len(corpus_texts)} articles")


---

## Section 1: LLM without RAG as baseline

Before adding RAG, let's see what the LLM knows on its own. We'll ask five questions of varying difficulty.

Keep a mental tally of which answers are correct, which are wrong, and which sound confidently wrong.

In [ ]:
import textwrap

QUESTIONS = [
    "In what year did the American Civil War end?",                                    # Q1: easy
    "Who was the first president of the United States?",                               # Q2: easy
    "What was the purpose of the Reconstruction era after the Civil War?",             # Q3: medium
    "What specific event in 1972 triggered the Watergate scandal?",                    # Q4: medium-hard
    "Name three specific programs established under the New Deal and what each did.",  # Q5: hard
]

baseline_answers = []
for i, q in enumerate(QUESTIONS, 1):
    print(f"\n{'='*80}\nQ{i}: {q}\n{'-'*80}")
    answer = generate(q, max_new_tokens=200)
    wrapped_text = textwrap.fill(answer, width=80)
    print(wrapped_text)
    baseline_answers.append(answer)


**Your turn:** look at each answer above and jot down whether it seems correct, partially correct, or wrong/hallucinated. We'll compare these to the RAG answers at the end.

*(It is okay to use a larger model, e.g. Gemini Thinking, to rate the above answers, but do make some notes here.)*

---

## Section 2: Chunking

We can't just embed each Wikipedia article as a single vector: the articles are too long (tens of thousands of characters each), and a single embedding for that much text loses all specificity. Instead, we **chunk** each article into smaller pieces and embed each chunk. We have to be careful about the chunk size though.

- Chunks too big, the embedding becomes a fuzzy average of many ideas, retrieval returns chunks that are vaguely related to everything but specifically related to nothing.
- Chunks too small, each chunk lacks context, retrieval returns fragments that mention the right word but miss the surrounding meaning.

A common starting point is 256 tokens with 32 tokens of overlap between adjacent chunks. The overlap helps queries that span chunk boundaries.

### TODO 0: implement fixed-size chunking

Fill in `chunk_text` below. The function should split `text` into overlapping windows measured in words (not tokens — close enough for our purposes and easier to implement).

In [ ]:
def chunk_text(text: str, chunk_size: int = 256, overlap: int = 32) -> list[str]:
    """Split `text` into overlapping word-based chunks.

    Args:
        text: the input string
        chunk_size: number of words per chunk
        overlap: number of words that adjacent chunks share

    Returns:
        list of chunk strings
    """
    # Hint: split the text into words, then slide a window of size `chunk_size`
    # advancing by `chunk_size - overlap` each step.

    # YOUR CODE HERE
    pass

# --- Test your implementation ---
sample = " ".join([f"word{i}" for i in range(600)])
chunks = chunk_text(sample, chunk_size=100, overlap=20)
print(f"Number of chunks: {len(chunks)}")   # expect ~8
print(f"First chunk ends with: ...{chunks[0].split()[-20:]}")
print(f"Second chunk starts with: {chunks[1].split()[:20]}...")
# The last 20 words of chunk 0 should equal the first 20 words of chunk 1.


### Apply chunking to the corpus

Now chunk every article and build two parallel lists:
- `chunks`: the chunk text
- `chunk_sources`: the article each chunk came from (useful for debugging)

In [ ]:
chunks = []
chunk_sources = []

for title, text in corpus_texts.items():
    article_chunks = chunk_text(text, chunk_size=256, overlap=32)
    chunks.extend(article_chunks)
    chunk_sources.extend([title] * len(article_chunks))

print(f"Total chunks: {len(chunks)}")
print(f"Avg chunks per article: {len(chunks) / len(corpus_texts):.1f}")
print(f"\nExample chunk (first 300 chars):\n{chunks[0][:300]}...")


---

## Section 3: Embeddings and the vector store

An embedding model is a neural network trained so that semantically similar pieces of text map to nearby points in a high-dimensional space. `all-MiniLM-L6-v2` maps any text to a 384-dimensional vector.

Concretely, if we embed "The Civil War ended in 1865" and "The war between the states concluded in 1865", the two vectors should have high cosine similarity — even though they share very few words.

### TODO 1: embed the corpus

Call `embedder.encode(...)` on the list of chunks. Pass `batch_size=64` and `show_progress_bar=True`. Time how long it takes.

In [ ]:
t0 = time.time()

# YOUR CODE HERE: embed `chunks` into a numpy array called `chunk_embeddings`
# Shape should be (len(chunks), 384). Use embedder.encode(...).
chunk_embeddings = None

embed_time = time.time() - t0

# --- Inspect the result ---
print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"Dtype: {chunk_embeddings.dtype}")
print(f"Memory footprint: {chunk_embeddings.nbytes / 1e6:.2f} MB")
print(f"Time to embed {len(chunks)} chunks: {embed_time:.1f}s")
print(f"Per-chunk: {embed_time / len(chunks) * 1000:.1f} ms")

### TODO 2: verify that embeddings capture meaning

We claimed that semantically similar sentences have high cosine similarity. Let's test it. Fill in `cosine_sim` and run the comparison below.

Recall that cosine similarity between vectors $a$ and $b$ is:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

In [ ]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors."""
    # YOUR CODE HERE
    pass

# --- Test with two pairs ---
pair_similar = (
    "The Civil War ended in 1865.",
    "The war between the states concluded in eighteen sixty-five.",
)
pair_unrelated = (
    "The Civil War ended in 1865.",
    "Bananas are a popular yellow fruit.",
)

for label, (s1, s2) in [("Similar", pair_similar), ("Unrelated", pair_unrelated)]:
    e1 = embedder.encode(s1)
    e2 = embedder.encode(s2)
    sim = cosine_sim(e1, e2)
    print(f"{label:10s}  sim={sim:.3f}")
    print(f"  s1: {s1}")
    print(f"  s2: {s2}\n")

# Expected: the similar pair should score noticeably higher (~0.6+)
# than the unrelated pair (~0.1 or lower).


---

## Section 4: Similarity search with brute-force k-NN

Now the core retrieval step. Given a query string, we want to find the top-k most similar chunks in the corpus. Without an index, we have to use brute force to find the top-k.

1. Embed the query
2. Compute cosine similarity between the query vector and *every* chunk vector
3. Return the top-k

For small corpora, this is fast. A single NumPy matrix multiplication computes all N similarities at once.

### Optimization trick: pre-normalize

If we L2-normalize both the query and the chunk embeddings ahead of time, then cosine similarity is just a dot product:

$$\cos(q, c) = \hat q \cdot \hat c$$

This turns the entire search into one `matmul` call.

In [ ]:
# Pre-normalize all chunk embeddings once
chunk_norms = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
chunk_embeddings_normed = chunk_embeddings / chunk_norms
print(f"Normalized embeddings shape: {chunk_embeddings_normed.shape}")
# Sanity check: each row should have L2 norm ≈ 1.0
print(f"Row norms (first 3): {np.linalg.norm(chunk_embeddings_normed[:3], axis=1)}")


### TODO: implement brute-force retrieval

Fill in `retrieve` below. It should return the top-k chunks and their similarity scores.

In [ ]:
def retrieve(query: str, k: int = 3) -> list[tuple[float, str, str]]:
    """Brute-force top-k retrieval.

    Returns:
        list of (score, source_title, chunk_text) tuples, sorted by score descending.
    """
    # Step 1: embed and normalize the query
    # YOUR CODE HERE
    query_vec = None

    # Step 2: compute cosine similarity against every chunk
    # Hint: one numpy matmul with chunk_embeddings_normed
    # YOUR CODE HERE
    scores = None

    # Step 3: find the top-k indices
    # Hint: np.argsort gives ascending order; use [::-1] to reverse, or np.argpartition for speed
    # YOUR CODE HERE
    top_k_idx = None

    # Step 4: assemble the result
    return [(float(scores[i]), chunk_sources[i], chunks[i]) for i in top_k_idx]

# --- Test on one of the questions the baseline LLM got wrong ---
test_query = "What specific event in 1972 triggered the Watergate scandal?"
results = retrieve(test_query, k=3)

print(f"Query: {test_query}\n")
for score, source, chunk in results:
    print(f"  [score={score:.3f}] from '{source}'")
    print(f"  {chunk[:250]}...\n")


Sanity check: the top result should come from the "Watergate scandal" article and should mention the June 1972 break-in at the DNC headquarters. If it doesn't, something is off with your implementation.

---

## Section 5: Putting it together

Now we combine retrieval and generation. The standard RAG pattern:

1. Retrieve top-k chunks for the query
2. Stuff them into a prompt as "context"
3. Ask the LLM to answer using only that context

The prompt template matters. We'll tell the model explicitly to ground its answer in the provided context and to admit if the context doesn't contain the answer. This reduces hallucination.

### TODO: build the RAG pipeline

In [ ]:
RAG_PROMPT_TEMPLATE = """You are answering a question using only the context provided below.
If the context does not contain the answer, say so.

Context:
{context}

Question: {question}

Answer:"""

def rag_answer(query: str, k: int = 3, max_new_tokens: int = 180) -> tuple[str, list]:
    """Full RAG pipeline: retrieve, build prompt, generate.

    Returns:
        (answer_text, retrieval_results)
    """
    # Step 1: retrieve top-k chunks
    # YOUR CODE HERE
    results = None

    # Step 2: concatenate the retrieved chunks into a single context string
    # Format suggestion: "[1] <chunk1>\n\n[2] <chunk2>\n\n[3] <chunk3>"
    # YOUR CODE HERE
    context = None

    # Step 3: fill in the prompt template and generate
    # YOUR CODE HERE
    prompt = None
    answer = None

    return answer, results


### Run all five questions through RAG and compare

This cell runs each question through the RAG pipeline and prints the answer alongside the baseline answer from Section 1.

In [ ]:
import textwrap

print("="*80)
print("BASELINE vs RAG COMPARISON")
print("="*80)

for i, q in enumerate(QUESTIONS, 1):
    rag_ans, _ = rag_answer(q)
    print(f"\n{'─'*80}\nQ{i}: {q}\n{'─'*80}")
    wrapped_text = textwrap.fill(baseline_answers[i-1], width=80)
    print(f"BASELINE (no retrieval):\n  {wrapped_text}\n")
    wrapped_text = textwrap.fill(rag_ans, width=80)
    print(f"RAG (with retrieval):\n  {wrapped_text}")

**Observe:** for which questions did RAG help the most? Were there any questions where RAG *hurt* (retrieved irrelevant context and made the model worse)?

Common pattern is that the easy factual questions may be unchanged since the model already knew the answer. The specific-detail questions usually show the biggest improvement — especially Q5, where the retrieved chunks contain concrete New Deal program names.

---

## Section 6: Will it scale?

RAG works. But will it scale? Let's observe if we even need vector databases.

### Exercise 6a: Time it

Instrument one RAG query end-to-end and measure how long each stage takes. Fill in the timing below.

In [ ]:
import time

query = "What specific event in 1972 triggered the Watergate scandal?"

# Step 1: time the query embedding
# YOUR CODE HERE
t0 = time.time()
# ... embed + normalize the query ...
t_embed = time.time() - t0

# Step 2: time the similarity search
# YOUR CODE HERE
t0 = time.time()
# ... compute scores and find top-k ...
t_search = time.time() - t0

# Step 3: time the LLM generation
# (Easiest: just call rag_answer and time it, then subtract embed + search.
#  Or re-use the retrieved chunks and call generate() directly.)
# YOUR CODE HERE
t0 = time.time()
# ... generate the answer ...
t_generate = time.time() - t0

total = t_embed + t_search + t_generate
print(f"Query embedding:   {t_embed*1000:7.1f} ms   ({t_embed/total*100:5.1f}%)")
print(f"Similarity search: {t_search*1000:7.1f} ms   ({t_search/total*100:5.1f}%)")
print(f"LLM generation:    {t_generate*1000:7.1f} ms   ({t_generate/total*100:5.1f}%)")
print(f"{'─'*50}")
print(f"Total:             {total*1000:7.1f} ms")


**Expected result:** with our ~1,000-chunk corpus, which is quite small, LLM time dominates. This makes RAG look cheap. But watch what happens as the corpus grows.

### Exercise 6b: search latency vs. corpus size

Let's simulate a larger corpus by duplicating our embedding matrix. We'll measure search time for corpus sizes from 1x to 1000x the original, then plot the result.

In [ ]:
multipliers = [1, 10, 100, 1000]

# Pre-embed and normalize the test query once
test_q = "What specific event in 1972 triggered the Watergate scandal?"
q_vec = embedder.encode(test_q)
q_vec = q_vec / np.linalg.norm(q_vec)

search_times = []
corpus_sizes = []

for m in multipliers:
    # Tile the normalized embedding matrix to simulate a corpus m× larger
    big_corpus = np.tile(chunk_embeddings_normed, (m, 1))
    n = big_corpus.shape[0]
    corpus_sizes.append(n)

    # Time the search (average over 5 runs)
    n_trials = 5
    t0 = time.time()
    for _ in range(n_trials):
        scores = big_corpus @ q_vec
        top_k = np.argpartition(-scores, 3)[:3]
    elapsed = (time.time() - t0) / n_trials
    search_times.append(elapsed)

    mem_gb = big_corpus.nbytes / 1e9
    print(f"  N={n:>10,}   search={elapsed*1000:7.2f} ms   memory={mem_gb:6.3f} GB")

    # Free memory before next iteration
    del big_corpus


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(corpus_sizes, [t*1000 for t in search_times], "o-", linewidth=2, markersize=8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Corpus size (number of chunks)")
ax.set_ylabel("Search latency (ms)")
ax.set_title("Brute-force k-NN: latency scales linearly with corpus size")
ax.grid(True, which="both", alpha=0.3)
# Reference line: 100 ms interactive-latency budget
ax.axhline(100, linestyle="--", color="red", alpha=0.6, label="100 ms interactive budget")
ax.legend()
plt.tight_layout()
plt.show()


### Exercise 6c: back-of-envelope for SPACEV1B

The SPACEV1B benchmark has 1.4 billion vectors, each 100-dimensional, stored as int8 (1 byte per dimension).

Use your measurements above to estimate:

1. How long would one brute-force query take on the full 1.4B-vector corpus?
   Use your measured scaling (linear in N) to extrapolate from the largest size you tested.

2. How much RAM would it take just to hold the vector matrix?

3. Compare to the 100 ms interactive-latency budget. By what factor is brute force too slow?

In [ ]:
# YOUR ANALYSIS HERE — fill in the numbers

# Estimated per-vector cost from your measurements (take the largest corpus):
time_per_vector_sec = None   # = search_times[-1] / corpus_sizes[-1]

# 1. Extrapolated latency for 1.4B vectors
N_SPACEV1B = 1_400_000_000
est_latency_sec = None       # = time_per_vector_sec * N_SPACEV1B

# 2. Memory for 1.4B × 100 dims × 1 byte
est_memory_gb = None         # = N_SPACEV1B * 100 * 1 / 1e9

# 3. Slowdown factor vs. 100 ms budget
slowdown_factor = None       # = est_latency_sec / 0.1

print(f"Estimated brute-force latency on 1.4B vectors:  {est_latency_sec:.1f} seconds")
print(f"Memory required just for vectors:               {est_memory_gb:.0f} GB")
print(f"Factor over 100 ms interactive budget:          {slowdown_factor:.0f}x")


## Takeaway

Brute-force k-NN is linear in corpus size. That's fine for 1,000 chunks, but doesn't work for a billion. This is why we need vector databases with clustering and graph indices.

**Comparison of vector databases:**

| | Brute force | IVF (clustering) | HNSW (graph) |
|---|---|---|---|
| Recall | 100% | ~95% (tunable) | ~99% (tunable) |
| Query latency at 1B vectors | minutes | ~ms | ~ms |
| Build time | zero | moderate | slow |
| Memory overhead | none | small | large (graph edges) |
| Supports updates | trivially | hard | hard |

In a real system, you'd replace the brute-force `retrieve` function with `faiss`, `hnswlib`, or a managed service (Pinecone, Weaviate, Milvus). The outer pipeline will also need to be productionized.